# Yeu cau 1

- Sử dụng bộ Phân lớp Linear Classifer: LogisticRegression
- Dùng phương pháp GridSearchCV sao cho kết quả cao nhất theo độ độ macro-F1:

In [6]:
import os

path = r'D:\Projects\Assignment\Courses\CS231-ICV\HoaVietNam2026\HoaVietNam2026'
train_path = r'D:\Projects\Assignment\Courses\CS231-ICV\HoaVietNam2026\HoaVietNam2026\train'
os.listdir(train_path)

['Cuc', 'Dao', 'Ga', 'Lan', 'Mai', 'Sen', 'Tho']

In [7]:
from PIL import Image 
import cv2
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression , SGDClassifier 

In [8]:
def extract_hsv_histogram(image):

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    hist = cv2.calcHist(
        [hsv],
        [0,1,2],
        None,
        [8,8,8],
        [0,180,0,256,0,256]
    )

    hist = cv2.normalize(hist, hist).flatten()

    return hist

In [9]:
flo_names = ['Cuc', 'Dao', 'Ga', 'Lan', 'Mai', 'Sen', 'Tho']
name_to_num = {name :  i for i , name  in enumerate(flo_names)}

def creat_dataset(type : str = 'train', return_img = False):

    type_path = os.path.join(path, type)

    features = []
    labels = []
    images = []

    print(f"XU LY TAP {type}")

    print("="* 50)

    for name in flo_names:
        print(f"Dang xu li folder {name}")
        for file_name in os.listdir(os.path.join(type_path , name)):
            
            src = os.path.join(type_path , name , file_name)

            img = cv2.imread(src)

            if img is None:
                continue
            
            feature = extract_hsv_histogram(img)
       

            features.append(feature)
            labels.append(name_to_num[name])
            images.append(img)
                
    if return_img == True:
        return features , labels , images
    return features , labels

In [10]:
X_train , y_train = creat_dataset('train')
X_test , y_test = creat_dataset('test')

XU LY TAP train
Dang xu li folder Cuc
Dang xu li folder Dao
Dang xu li folder Ga
Dang xu li folder Lan
Dang xu li folder Mai
Dang xu li folder Sen
Dang xu li folder Tho
XU LY TAP test
Dang xu li folder Cuc
Dang xu li folder Dao
Dang xu li folder Ga
Dang xu li folder Lan
Dang xu li folder Mai
Dang xu li folder Sen
Dang xu li folder Tho


In [18]:
logis_classifier = LogisticRegression(max_iter=1000)

param_grid = [
    {
        "solver": ["lbfgs", "newton-cg", "newton-cholesky", "sag"],
        "penalty": ["l2", None]
    },
    {
        "solver": ["liblinear"],
        "penalty": ["l1", "l2"]
    },
    {
        "solver": ["saga"],
        "penalty": ["l1", "l2", "elasticnet", None]
    }
]

grid_cv = GridSearchCV(
    estimator=logis_classifier,
    param_grid=param_grid,
    scoring="f1_macro",
    verbose=1
)

grid_cv.fit(X_train, y_train)

print(grid_cv.best_params_)

Fitting 5 folds for each of 14 candidates, totalling 70 fits


d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avo

{'penalty': 'l2', 'solver': 'lbfgs'}


d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
10 fits failed out of a total of 70.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Projects\Assignment\Courses\venv\Lib\site-packages\sklearn\base.py", line 1336, in wrapp

In [19]:
y_pred = grid_cv.predict(X_test)

print(classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.54      0.70      0.61        10
           1       0.77      0.83      0.80        12
           2       0.85      0.73      0.79        15
           3       0.80      0.40      0.53        10
           4       0.80      0.80      0.80        10
           5       0.54      0.70      0.61        10
           6       0.80      0.80      0.80        10

    accuracy                           0.71        77
   macro avg       0.73      0.71      0.71        77
weighted avg       0.74      0.71      0.71        77



# Yeu cau 2

Thực hiện tương tự như yêu cầu 1 nhưng sử dụng phương pháp SGDClassifier 

In [20]:
logis_classifier = SGDClassifier(random_state=42)

param_grid = {
    'loss': ['hinge', 'log_loss', 'modified_huber' , 'perceptron' , 'squared_hinge'],
    'penalty': ['l2', 'l1', 'elasticnet'],
    'learning_rate': ['optimal', 'invscaling', 'adaptive'],
    'early_stopping': [True, False],
    'alpha': [1e-4, 1e-3, 1e-2]
}

grid_cv = GridSearchCV(
    estimator=logis_classifier,
    param_grid=param_grid,
    scoring="f1_macro",
    verbose=1,
    n_jobs=-1,
    cv = 3
)

grid_cv.fit(X_train, y_train)

print(grid_cv.best_params_)
print(grid_cv.best_score_)

Fitting 3 folds for each of 270 candidates, totalling 810 fits
{'alpha': 0.01, 'early_stopping': False, 'learning_rate': 'adaptive', 'loss': 'modified_huber', 'penalty': 'l2'}
0.750655839251522


In [21]:
y_pred = grid_cv.predict(X_test)

print(classification_report(y_test , y_pred))

              precision    recall  f1-score   support

           0       0.54      0.70      0.61        10
           1       0.77      0.83      0.80        12
           2       0.86      0.80      0.83        15
           3       0.83      0.50      0.62        10
           4       0.89      0.80      0.84        10
           5       0.58      0.70      0.64        10
           6       0.80      0.80      0.80        10

    accuracy                           0.74        77
   macro avg       0.75      0.73      0.73        77
weighted avg       0.76      0.74      0.74        77

